In [1]:
print(123)

123


In [2]:
print("Hello, World!")

Hello, World!


In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from openai import OpenAI
openai_client  = OpenAI()

In [5]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [6]:
llm("hello, what day is it today?")

'Today is **Saturday, July 19, 2025**.'

In [7]:
question = "i just heard about the LLM course, can i still join it?"
response = llm(question)
print(response)

Maybe — but it depends on the course policy and how far along it is.

A good next step is to check:
- whether enrollment is still open
- if there’s a waitlist
- whether recordings/materials are available
- if late enrollment is allowed

If you want, I can help you draft a short message to the course instructor or organizer asking if you can still join.


In [8]:
context = """I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting 
submissions.

#Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework 
(while the form is open) without registering. It is not checked against any registered list. 
Registration is just to gauge interest before the start date.

#What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). 
The video URL should be posted in the announcements channel on Telegram and Slack before it begins. 
You can also watch live on the DataTalksClub YouTube Channel.

Don’t post questions in chat as they may be missed if the room is very active.

#How should I start the course and follow the weekly workflow?
Start with the LLM Zoomcamp docs, the general Zoomcamp logistics docs, and the LLM Zoomcamp GitHub repository.

You can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed 
in the course management platform.

A typical workflow is:

Watch the lesson videos.
Work through the lesson notebooks/code.
Read the homework instructions on GitHub.
Submit answers through the course platform before the deadline.
Homework is similar to the lesson flow, but uses a different dataset or slightly different task.

"""

In [9]:
prompt = f"""Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question: {question}

Context = {context}
"""

print(prompt)

Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question: i just heard about the LLM course, can i still join it?

Context = I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting 
submissions.

#Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework 
(while the form is open) without registering. It is not checked against any registered list. 
Registration is just to gauge interest before the start date.

#What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs.

S

In [10]:
response = llm(prompt)
print(response)

Yes, you can still join the course. If you want a certificate, make sure to submit your project while submissions are still open.


In [11]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [12]:
courses_raw

[{'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 471},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 253},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 404},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 118}]

In [13]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1380

In [14]:
documents[100]

{'id': '1fda7c57b0',
 'course': 'machine-learning-zoomcamp',
 'section': 'Module 2. Machine Learning for Regression',
 'question': 'ValueError: shapes not aligned',
 'answer': '```python\nX_train = prepare_X(df_train)\nw_0, w = train_linear_regression(X_train, y_train)\n\nX_val = prepare_X(df_val)\ny_pred = w_0 + X_val.dot(w)\n\nrmse(y_val, y_pred)\n```\n\nWe get:\n\n```\nValueError                                Traceback (most recent call last)\nInput In [132], in <cell line: 5>()\n      2 w_0, w = train_linear_regression(X_train, y_train)\n      4 X_val = prepare_X(df_val)\n----> 5 y_pred = w_0 + X_val.dot(w)\n      7 rmse(y_val, y_pred)\n\nValueError: shapes (4128,) and (1,) not aligned: 4128 (dim 0) != 1 (dim 0)\n```\n\nIf we try to perform an arithmetic operation between two arrays of different shapes or dimensions, it throws an error like operands could not be broadcast together with shapes. Broadcasting can occur in certain scenarios and will fail in others.\n\nTo solve this is

In [15]:
def rag(question):
    search_results = search(question)
    user_prompt = built_prompt(question, search_results)
    return llm(user_prompt)


In [16]:
from minsearch import Index

index = Index(
  text_fields = ["question", "answer", "section"],
  keyword_fields = ["course"]
 )

index.fit(documents)

In [17]:
def search(question, course="llm-zoomcamp"):

    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )
    
search(question)

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': 'a9353fadfe',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The homework submission form is still open even though the deadline has passed — can I still submit?',
  

In [18]:
search_results = search(question)

In [19]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [20]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:\n
{context} 
"""

In [21]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append("Section: " + doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()


In [22]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(question=question, context=context)

    return prompt.strip()

In [23]:
prompt = build_prompt(question, search_results)
print(prompt)

Question:
i just heard about the LLM course, can i still join it?

Context:

Section: General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

Section: General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

Section: General Course-Related Questions
Q: The homework submission form is still open even though the deadline has passed — can I still submit?
A: Yes. As long as the submission form is still open, you can submit your answers, even if the listed deadline has already passed. You can no longer submit only a

In [24]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

print(response)

Response(id='resp_0864047b2ba2233b006a5cb87c4f7c81a08db4af025bc3ad72', created_at=1784461436.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_0864047b2ba2233b006a5cb87c957881a0a1cf65684354c1ca', content=[ResponseOutputText(annotations=[], text='Yes — you can still join the LLM Zoomcamp.\n\nYou don’t need a registration confirmation email to start learning or submitting homework. Just begin the course and submit assignments while the submission form is still open.\n\nIf you want to receive a certificate, make sure to submit your project before submissions close.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=0.98, background=False, completed_at=1784461437.0, conversation=None, max_output_tokens=None, max_tool_calls=None, moderation

In [25]:
response.output[0]

ResponseOutputMessage(id='msg_0864047b2ba2233b006a5cb87c957881a0a1cf65684354c1ca', content=[ResponseOutputText(annotations=[], text='Yes — you can still join the LLM Zoomcamp.\n\nYou don’t need a registration confirmation email to start learning or submitting homework. Just begin the course and submit assignments while the submission form is still open.\n\nIf you want to receive a certificate, make sure to submit your project before submissions close.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')

In [26]:
response.output[0].content[0]

ResponseOutputText(annotations=[], text='Yes — you can still join the LLM Zoomcamp.\n\nYou don’t need a registration confirmation email to start learning or submitting homework. Just begin the course and submit assignments while the submission form is still open.\n\nIf you want to receive a certificate, make sure to submit your project before submissions close.', type='output_text', logprobs=[])

In [27]:
response.output[0].content[0].text

'Yes — you can still join the LLM Zoomcamp.\n\nYou don’t need a registration confirmation email to start learning or submitting homework. Just begin the course and submit assignments while the submission form is still open.\n\nIf you want to receive a certificate, make sure to submit your project before submissions close.'

In [28]:
response.output_text

'Yes — you can still join the LLM Zoomcamp.\n\nYou don’t need a registration confirmation email to start learning or submitting homework. Just begin the course and submit assignments while the submission form is still open.\n\nIf you want to receive a certificate, make sure to submit your project before submissions close.'

In [29]:
response.usage

ResponseUsage(input_tokens=392, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=64, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=456)

In [30]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.000582

In [31]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

reponse = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=message_history
)

In [32]:
reponse.output_text

'Yes, you can still join. If you want a certificate, you need to submit your project while submissions are still being accepted.'

In [33]:
def llm(instructions, prompt, model='gpt-5.4-mini'):
    message_history = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [39]:
def rag(query, model='gpt-5.4-mini'):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [41]:
answer = rag(question)
print(answer)

Yes — you can still join.

If you want a certificate, make sure to submit your project while submissions are still open.


In [42]:
answer = rag("when can i get the certificate for the course?")
print(answer)

You can get the certificate only if you finish the course with a live cohort and submit your capstone project while submissions are still open. You also need to complete the required peer reviews. Homework is not required.


In [43]:
answer = rag("how old is the course")
print(answer)

I don't know.
